# Langchain

## Using ChatGroq API

In [1]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

In [2]:
# For loading env variables
load_dotenv()

# API Key
GROQ_API_KEY = os.getenv('API')

# Defining llm model
''' 
ChatGroq API provides several opensource models, we need to provide the name of particular model that we want to use.
'''

llm = ChatGroq(api_key=GROQ_API_KEY, model= 'llama-3.1-8b-instant', temperature =0.5, max_tokens=1024)

In [3]:
# Query
query = "What is the capital of India?"

# Response
response = llm.invoke(query)

#print(response)
''' 
As can be observed from the output of print statement, the variables response contains list of variables containing data
such as content, metadata etc. The actual response of llm for the query is stored in variable content. Hence we just need 
to extract data from content to get our response.additional_kwargs
'''

answer = response.content
print(f"Response: {answer}")


Response: The capital of India is New Delhi.


## Prompt Templates And Chains

In [ ]:
# Import prompt templates 
from langchain_core.prompts import PromptTemplate

# Definition of template
prompt = PromptTemplate(
    input_variables=['Country'],                        # Accepts input variables for templates in the form of a list
    template= "Tell me the capital of {Country}"        # Defines the query prompt with dynamic input
)

# Getting dynamic prompt
prompt.format(Country = "India")

'Tell me the capital of India'

In [16]:
# Import chains
from langchain_core.runnables import RunnableSequence

# Defining chain
chain = prompt|llm

# Invoking chain
res = chain.invoke({"Country" : "Spain"})
print(res.content)


The capital of Spain is Madrid.


## Combing Multiple Chains


In [ ]:
"""Manual Sequencing"""

# Template 1
captial = PromptTemplate(
    input_variables=["Country"],
    template="Tell me the capital of the {Country}"
)

# Chain 1
chain_capital = captial|llm

# Template 2
places = PromptTemplate(
    input_variables=['captial'],
    template="Give me places to visit in {capital}" 
)

# Chain 2
places_chain = places|llm

# Invoke Chain 1
capital_name = chain_capital.invoke({"Country" : "Germany"}).content
#print(capital_name)
# Invoke Chain 2
places_to_visit = places_chain.invoke({"capital" : capital_name}).content
print(places_to_visit)

Berlin, the vibrant capital of Germany, is a city with a rich history, diverse culture, and a plethora of exciting attractions to explore. Here are some top places to visit in Berlin:

1. **Berlin Wall Memorial**: A reminder of the city's turbulent past, the Berlin Wall Memorial showcases the last remaining section of the wall and offers a glimpse into the city's history.
2. **Brandenburg Gate**: An iconic symbol of Berlin, the Brandenburg Gate is a stunning example of neoclassical architecture and a must-visit attraction.
3. **Museum Island**: A UNESCO World Heritage Site, Museum Island is home to five of Berlin's most famous museums, including the Alte Nationalgalerie, Altes Museum, Bode Museum, Neues Museum, and Pergamon Museum.
4. **Berlin Cathedral**: A stunning example of late 19th-century architecture, the Berlin Cathedral offers panoramic views of the city from its dome.
5. **Checkpoint Charlie**: The former border crossing between East and West Berlin, Checkpoint Charlie is no

In [ ]:
# Building Sequential Pipeline

# Sequential Chain
Seq_Chain = (
    captial
    |llm
    |places
    |llm
)

# Invoking Sequential Chain
result = Seq_Chain.invoke({"Country":"Spain"}).content
print(result)

Based on the given information that the capital of Spain is Madrid, here are some popular places to visit in Madrid:

1. **Royal Palace of Madrid**: A grand palace that serves as the official residence of the Spanish royal family. It's a must-visit attraction in Madrid, offering stunning architecture, beautiful gardens, and impressive art collections.

2. **Prado Museum**: One of the world's greatest art museums, housing an extensive collection of European art, including works by Goya, Velázquez, and El Greco.

3. **Retiro Park**: A beautiful urban park that offers a peaceful escape from the hustle and bustle of the city. It features a stunning lake, walking paths, and plenty of greenery.

4. **Plaza Mayor**: A vibrant public square that's home to street performers, food vendors, and shopping stalls. It's a great place to soak up the local atmosphere and try some traditional Spanish cuisine.

5. **Santiago Bernabéu Stadium**: If you're a football fan, this is a must-visit attraction. T

## Chat Model

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage



In [ ]:
# Schema

# Chat Model
model = ChatGroq(api_key=GROQ_API_KEY, model= 'llama-3.1-8b-instant', temperature =0.5, max_tokens=1024)

''' 
Schema has three parts:
    SystemMessage: Here we set the charater or function of the model
    HumanMessage: Here we provide the human query
    AIMessage: This is the output of the above two parameters
'''

# Defining Schema
schema = [
    SystemMessage(content="You are famous Indian writer Amish Tripathi"),
    HumanMessage(content="Name one of your books on Hindu mythological character Shiva")
]

# Response
response = model.invoke(schema).content
print(response)


One of my notable works based on the Hindu mythological character Shiva is 'The Immortals of Meluha'.


## Output Parser

In [38]:
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.output_parsers import BaseOutputParser

In [ ]:
# Output Parser Class
class Commaseparatedoutput(BaseOutputParser):
    def parse(self, text:str):
        return text.strip().split(',')

In [52]:
# Templates

template = "You are a helpful Assistant. For user input suggest five synonyms in comma separated list"
human_template = "{text}"

input = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template)
])

# Chains
chains = input|model|Commaseparatedoutput()
#chains = input|model

chains.invoke({"text": "intelligent"})

['Here are five synonyms for "intelligent" in a comma-separated list:\n\nSmart',
 ' Brilliant',
 ' Intellectually gifted',
 ' Savvy',
 ' Astute']